# Assignment 30: Chat Groq RAG Application with Streamlit UI

# PART 1 -- Groq Setup & Basic Chat

# Task 1 : ChatGroq Setup
## 1.Create a Groq account and generate an API key
## 2. Configure the API key using environment veriables
## 3. Initialize ChatGroq using Langchain.

In [1]:
## Loding env file
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
## Initialize ChatGroq
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = model.invoke("What is Generative AI?")
print(response.content)


**Generative AI** (short for *generative artificial intelligence*) is a class of machine‑learning models that can create new content—such as text, images, audio, video, code, or even 3‑D designs—rather than just analyzing or classifying existing data.

---

## Core Idea

- **Learning from data:** The model is trained on a large dataset of examples (e.g., millions of sentences, photos, or code snippets).  
- **Capturing patterns:** During training it learns statistical relationships, structures, and “rules” that underlie the data.  
- **Generating new samples:** After training, the model can produce novel outputs that resemble the training data but are not exact copies.

---

## Main Types of Generative Models

| Model family | Typical output | How it works (high‑level) |
|--------------|----------------|---------------------------|
| **Transformer‑based language models** (e.g., GPT‑4, LLaMA, Claude) | Text, code, conversational replies | Predict the next token (word/character) given th

# Task 2: Basic Chat with ChatGroq
## 1. Send a simple prompt to chatGroq.
## 2. Print the response
## 3. verify low-latency behavior

In [2]:
import time
## simple prompt 
prompt= "Explain Generative AI in 4 lines"
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
start = time.perf_counter()

response = model.invoke(prompt)
end = time.perf_counter()
print(response.content)

# Print latency
print(f"the start time {start},and the end time {end}")

Generative AI are models that learn patterns from existing data (text, images, audio, etc.) and then create new, original content that mimics those patterns.  
They use deep‑learning architectures—most commonly transformers or diffusion networks—to predict the next piece of data given a context.  
During training they adjust billions of parameters to minimize the difference between their outputs and real examples, enabling them to “imagine” plausible variations.  
When prompted, they sample from this learned distribution, producing text, images, music, code, or other media that can be indistinguishable from human‑crafted creations.
the start time 430.1421483,and the end time 431.1965623


# PART 2 -- RAG Backend Pipeline


# Task 3: Document Loading & Text Splitting
## 1. Load the documents using Langchain Loaders
## 2. Split documents using RecursiveCharacterTextSplitter
## 3. Configure:
## . Chunk_size
## . Chunk_overlap

In [3]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load PDF document
loader = PyPDFLoader("medical_health_data_document.pdf")
documents = loader.load()
print("Total pages:", len(documents))

# 2. Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# 3. Split documents into chunks
chunks = text_splitter.split_documents(documents)

# 4. Display results
print("Total chunks:", len(chunks))
print(chunks[0].page_content)


C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_3216\3650875102.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
incorrect startxref pointer(1)
parsing for Object Streams


Total pages: 4
Total chunks: 17
Medical Health Data: An Overview of Common
 Health Indicators
Educational document — 3–5 page reference material
This document provides general, educational information about commonly recorded medical health
indicators, preventive care, and basic interpretation of health data. It is intended for learning and does
not replace diagnosis or advice from a qualified healthcare professional.
1. Introduction to Medical Health Data


# Task 4: Embeddings & Vector Store
## 1. Generate embeddings
## 2. Store Embeddins in: FAISS or ChromaDB
## 3. Crete a retriever from the vector store



In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

## chunks
chunks = text_splitter.split_documents(documents)

## embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# 2. Generate embeddings and store in FAISS
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print("Documents stored in FAISS:", len(chunks))

# 3. Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# 4. Test the retriever
query = "What is the main topic of this document?"
results = retriever.invoke(query)

print("\nRetrieved documents:")
for i, doc in enumerate(results, start=1):
    print(f"\nDocument {i}:")
    print(doc.page_content)


Documents stored in FAISS: 17

Retrieved documents:

Document 1:
Medical Health Data: An Overview of Common
 Health Indicators
Educational document — 3–5 page reference material
This document provides general, educational information about commonly recorded medical health
indicators, preventive care, and basic interpretation of health data. It is intended for learning and does
not replace diagnosis or advice from a qualified healthcare professional.
1. Introduction to Medical Health Data

Document 2:
1. Introduction to Medical Health Data
Medical health data is information collected about a person's physical condition, symptoms,
measurements, laboratory results, medical history, medications, and preventive-care activities.
Healthcare professionals use these data to understand health status, identify possible risks, monitor
changes over time, and support clinical decisions.
Common health measurements include body temperature, blood pressure, heart rate, respiratory rate,

Document 3:
4.

# Task 5: RAG Prompt Template
## Create a ChatPromptTemplate that:
## Injects retrieved context
## Accepts user question
## Instructs the model to answer only from context

In [5]:

from langchain_core.prompts import ChatPromptTemplate

#  prompt template
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using ONLY the
provided context.

If the answer is not present in the
context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
""")


# PART 3 - ChatGroq RAG Chain

# Task 6: Build RAG Chain
## Create a pipeline: user Question ---> Retriever ---> Context ---> Prompt---> ChatGroq ---> Answer 
## Test the chain with multiple questions.

In [10]:

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

#  Load PDF
loader = PyPDFLoader("medical_health_data_document.pdf")
documents = loader.load()

#  Split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)

#  Create embeddings and FAISS vector store

## embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

#  Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

#  Initialize ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

#  prompt template
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using ONLY the
provided context.

If the answer is not present in the
context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
""")

#  Build RAG chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda question: question
    }
    | prompt
    | llm
    | StrOutputParser()
)
# Test with multiple questions
questions = [
    "What is the main topic of this document?",
    "What are the key points discussed?",
    "Summarize the document briefly."
]

for question in questions:
    print("\nQuestion:", question)
    answer = rag_chain.invoke(question)
    print("Answer:", answer)


incorrect startxref pointer(1)
parsing for Object Streams



Question: What is the main topic of this document?
Answer: The main topic of this document is medical health data and common health indicators.

Question: What are the key points discussed?
Answer: **Key points discussed**

1. **When medical data warrants professional review**  
   - Unusual, repeatedly abnormal, rapidly changing measurements, or those accompanied by concerning symptoms should be evaluated by a professional.  
   - Emergency symptoms (severe breathing difficulty, severe chest pain, sudden weakness or confusion, uncontrolled bleeding, loss of consciousness) require urgent medical attention.

2. **Blood pressure and cardiovascular health**  
   - Blood pressure is recorded as systolic over diastolic (mmHg).  
   - Systolic reflects pressure during heart contraction; diastolic reflects pressure between beats.  
   - Repeated measurements are more informative than a single reading because blood pressure can vary with stress, activity, caffeine, illness, medications, and m

# PART 4 -- Streamlit UI

# Task 7: Build Streamlit Chat Interface
## Create a streamlit app with:
## 1. File uploader(PDF/Text)
## 2. Chat input box
## 3. Chat message display(user & AI)
## 4. Session state to store chat history

# Task 8: Integrate RAG Backend with UI
## 1. Connect Streamlit UI with RAG chain
## 2. ON user query:
## .Retrieve relevent documents.
## .Generate answer using chatGroq
## 3. Display responses in chat format

# PART 5-- Testing And Validation

# TAsk 9: Multi - Turn Chat Testing
## Test the chatbot with:
## 1. Initial factual question
## 2. Follow-up questtion
## 3. Out of Context question
## verify:
## Answers are grounded in documnets
## Chat history is maintained

# PART 6--Mini Project Deliverable

# Task 10: Final ChatGroq RAG App
## Ensure your app:
## Loads documents dynamically
## Responds quickly using groq
## Maintains conversation history
## Handles errors gracefully

# Task 11: Observations & Insights
## Write short answers
## 1. Why groq is suitable for RAG chatbots
## 2. Difference between Groq RAG and OpenAI RAG
## 3. Role of Streamlit in rapid GenAI prototyping

## 1. Why Groq is suitable for RAG chatbots
### Groq provides fast LLM inference, low response latency, and quick answers, making it suitable for interactive RAG chatbots.

## 2. Difference between Groq RAG and OpenAI RAG
### Groq RAG uses Groq's inference platform and supported models, while OpenAI RAG uses OpenAI's models and API. Both can retrieve relevant documents and generate context-based answers; they differ in models, pricing, speed, and API setup.

## 3. Role of Streamlit in rapid GenAI prototyping
### Streamlit helps developers quickly build interactive web apps using Python. It provides simple components for file uploads, chat interfaces, displaying responses, and testing GenAI applications without extensive frontend development.